In [ ]:
import csv

GROUP_ROW_START = 33
GROUP_ROW_END   = 44


# 사용자 정의 예외
class BirthdayMissingError(Exception):
    pass


# Node
class Node:
    def __init__(self, name, birthday):
        self.name     = name
        self.birthday = birthday   # 문자열 or None
        self.prev     = None
        self.next     = None


# CircularDoublyLinkedList
class CircularDoublyLinkedList:
    def __init__(self):
        self.head = None
        self.size = 0

    def is_empty(self):
        return self.head is None

    def insert_tail(self, name, birthday):
        new_node = Node(name, birthday)
        if self.is_empty():
            new_node.prev = new_node
            new_node.next = new_node
            self.head = new_node
        else:
            tail          = self.head.prev   # 마지막 노드
            tail.next     = new_node
            new_node.prev = tail
            new_node.next = self.head
            self.head.prev = new_node
        self.size += 1

    def display_forward(self):
        # 순방향 출력 (head -> tail)
        if self.is_empty():
            print("  (리스트가 비어 있습니다)")
            return
        cur  = self.head
        rank = 1
        while True:
            bday_str = cur.birthday if cur.birthday else "생년월일 없음"
            print(f"  {rank:>2}. {cur.name:<12} {bday_str}")
            rank += 1
            cur = cur.next
            if cur is self.head:
                break

    def display_reverse(self):
        # 역방향 출력 (tail -> head)
        if self.is_empty():
            print("  (리스트가 비어 있습니다)")
            return
        cur  = self.head.prev   # tail
        rank = 1
        while True:
            bday_str = cur.birthday if cur.birthday else "생년월일 없음"
            print(f"  {rank:>2}. {cur.name:<12} {bday_str}")
            rank += 1
            cur = cur.prev
            if cur is self.head.prev:
                break



# CSV에서 같은 조
def load_group_members(filepath, row_start, row_end):
    members = []
    with open(filepath, encoding="utf-8-sig", newline="") as f:
        reader = csv.reader(f)
        for cell_row, row in enumerate(reader, start=1):
            if cell_row < row_start or cell_row > row_end:
                continue
            if not row or row[0].strip() == "":
                continue
            name = row[0].strip()
            try:
                if len(row) < 4 or row[1].strip() == "" or row[2].strip() == "" or row[3].strip() == "":
                    raise BirthdayMissingError(f"'{name}': 생년월일 데이터 없음")
                year  = int(float(row[1]))
                month = int(float(row[2]))
                day   = int(float(row[3]))
                bday  = f"{year}년 {month:02d}월 {day:02d}일"
                members.append((name, bday))
            except BirthdayMissingError as e:
                print(f"  [예외처리] {e} -> birthday=None 으로 저장")
                members.append((name, None))
            except ValueError:
                print(f"  [예외처리] '{name}': 형식 오류 -> birthday=None 으로 저장")
                members.append((name, None))
    return members


# 메인
def main():
    filepath = "C:/Users/유가현/Downloads/birthday.csv.csv"

    members = load_group_members(filepath, GROUP_ROW_START, GROUP_ROW_END)

    if not members:
        print("조원 데이터를 불러오지 못했습니다.")
        return

    # 원형 이중 연결 리스트 구성
    cdll = CircularDoublyLinkedList()
    for name, bday in members:
        cdll.insert_tail(name, bday)

    print(f"\n  총 {cdll.size}명 리스트 삽입 완료")

    # 순방향 출력
    print(f"\n  ▶ 순방향 출력")
    print("  " + "-" * 46)
    cdll.display_forward()

    # 역방향 출력
    print(f"\n  ◀ 역방향 출력")
    print("  " + "-" * 46)
    cdll.display_reverse()

    print("=" * 50)


if __name__ == "__main__":
    main()
